# Multi-Model RAG Evaluation

This notebook evaluates RAG performance across all embedding models:
1. OpenAI text-embedding-3-small
2. Mistral mistral-embed
3. Snowflake snowflake-arctic-embed-l
4. BAAI bge-large-en-v1.5

And generates comparison reports and visualizations.

In [ ]:
# Setup
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / '.env')

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime

from ragflow_client import RAGFlowClient
from evaluation_pipeline import EvaluationPipeline
from utils import load_dataset_from_jsonl, format_results_table
from config import EMBEDDING_MODELS

print("✓ Imports successful")

## 1. Load Test Dataset

In [ ]:
# Load dataset
dataset_path = Path.cwd().parent / 'data' / 'test_qa_pairs.jsonl'
dataset = load_dataset_from_jsonl(str(dataset_path))

print(f"📊 Loaded {len(dataset)} test questions")
print(f"🔬 Testing {len(EMBEDDING_MODELS)} embedding models")
print(f"📈 Total queries: {len(dataset) * len(EMBEDDING_MODELS)}")

## 2. Review Models to Evaluate

In [ ]:
# Display models
models_df = pd.DataFrame(EMBEDDING_MODELS)
print("\nEmbedding Models:")
print("=" * 100)
display(models_df[['display_name', 'provider', 'model', 'dimensions', 'description']])
print("=" * 100)

## 3. Initialize Pipeline

In [ ]:
# Initialize RAGFlow client
client = RAGFlowClient()

# Initialize evaluation pipeline
pipeline = EvaluationPipeline(
    ragflow_client=client,
    evaluator_model='gpt-4o-mini'  # LLM for Ragas metrics
)

print("✓ Pipeline initialized")
print("✓ Ragas evaluator: gpt-4o-mini")

## 4. Run Multi-Model Evaluation

This will:
- Query each of the 20 questions with each embedding model
- Collect responses and retrieved contexts
- Run Ragas evaluation for each model
- Save individual results

**Note:** This may take 10-15 minutes depending on API speed.

In [ ]:
# Create timestamped results directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_dir = Path.cwd().parent / 'results' / f'multi_model_{timestamp}'

print(f"Results will be saved to: {results_dir}\n")
print("="*80)
print("STARTING MULTI-MODEL EVALUATION")
print("="*80)

In [ ]:
# Run evaluation
all_results = pipeline.run_multi_model_evaluation(
    test_dataset=dataset,
    model_configs=EMBEDDING_MODELS,
    save_dir=str(results_dir)
)

## 5. Generate Comparison Table

In [ ]:
# Collect results into comparison table
comparison_data = []

for model_config in EMBEDDING_MODELS:
    model_name = model_config['name']
    result = all_results[model_name]
    
    if result['success'] and result['ragas_results'] is not None:
        # Get metrics
        df = result['ragas_results'].to_pandas()
        metrics = df.mean()
        
        row = {
            'Model': model_config['display_name'],
            'Provider': model_config['provider'],
            'Dimensions': model_config['dimensions'],
            'Context Precision': metrics.get('context_precision', 0),
            'Context Recall': metrics.get('context_recall', 0),
            'Faithfulness': metrics.get('faithfulness', 0),
            'Answer Relevancy': metrics.get('answer_relevancy', 0),
        }
        comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*100)
print("COMPARISON TABLE: All Embedding Models")
print("="*100)
display(comparison_df.round(3))
print("="*100)

In [ ]:
# Save comparison table
comparison_path = results_dir / 'comparison_table.csv'
comparison_df.to_csv(comparison_path, index=False)
print(f"\n💾 Saved comparison table to: {comparison_path}")

## 6. Identify Best Performing Model

In [ ]:
# Find best model for each metric
print("\n🏆 BEST PERFORMING MODELS BY METRIC")
print("="*80)

metrics = ['Context Precision', 'Context Recall', 'Faithfulness', 'Answer Relevancy']

for metric in metrics:
    best_idx = comparison_df[metric].idxmax()
    best_model = comparison_df.loc[best_idx, 'Model']
    best_score = comparison_df.loc[best_idx, metric]
    print(f"{metric:20s}: {best_model:40s} ({best_score:.3f})")

print("="*80)

# Calculate overall average
comparison_df['Overall Average'] = comparison_df[metrics].mean(axis=1)
best_overall_idx = comparison_df['Overall Average'].idxmax()
best_overall = comparison_df.loc[best_overall_idx, 'Model']
best_overall_score = comparison_df.loc[best_overall_idx, 'Overall Average']

print(f"\n🎯 BEST OVERALL MODEL: {best_overall}")
print(f"   Average Score: {best_overall_score:.3f}")
print("="*80)

## 7. Visualizations

In [ ]:
# Bar chart comparing all metrics
fig = go.Figure()

for metric in metrics:
    fig.add_trace(go.Bar(
        name=metric,
        x=comparison_df['Model'],
        y=comparison_df[metric],
        text=comparison_df[metric].round(3),
        textposition='auto',
    ))

fig.update_layout(
    title='RAG Performance Comparison: All Metrics by Embedding Model',
    xaxis_title='Embedding Model',
    yaxis_title='Score',
    yaxis_range=[0, 1],
    barmode='group',
    height=600,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

# Save figure
fig.write_html(str(results_dir / 'comparison_chart.html'))
print(f"\n💾 Saved chart to: {results_dir / 'comparison_chart.html'}")

In [ ]:
# Radar chart for overall comparison
fig = go.Figure()

for _, row in comparison_df.iterrows():
    fig.add_trace(go.Scatterpolar(
        r=[row['Context Precision'], row['Context Recall'], 
           row['Faithfulness'], row['Answer Relevancy']],
        theta=['Context Precision', 'Context Recall', 'Faithfulness', 'Answer Relevancy'],
        fill='toself',
        name=row['Model']
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 1]
        )),
    showlegend=True,
    title='Radar Chart: Embedding Model Comparison',
    height=600
)

fig.show()

# Save figure
fig.write_html(str(results_dir / 'radar_chart.html'))
print(f"💾 Saved radar chart to: {results_dir / 'radar_chart.html'}")

## 8. Per-Question Analysis

In [ ]:
# Find which model performed best for each question
print("\n📊 PER-QUESTION PERFORMANCE ANALYSIS")
print("="*100)

# Collect per-question scores
per_question_data = []

for i in range(len(dataset)):
    question = dataset[i]['user_input'][:50] + '...'
    
    for model_config in EMBEDDING_MODELS:
        model_name = model_config['name']
        result = all_results[model_name]
        
        if result['success'] and i < len(result['eval_data']):
            df = result['ragas_results'].to_pandas()
            if i < len(df):
                row_data = df.iloc[i].to_dict()
                row_data['Question'] = question
                row_data['Model'] = model_config['display_name']
                per_question_data.append(row_data)

per_question_df = pd.DataFrame(per_question_data)

# Show summary
if not per_question_df.empty:
    print(f"\nCollected {len(per_question_df)} data points")
    display(per_question_df.head(10))
else:
    print("No per-question data available")

## 9. Statistical Summary

In [ ]:
# Statistical significance tests (if needed)
print("\n📈 STATISTICAL SUMMARY")
print("="*100)

for metric in metrics:
    print(f"\n{metric}:")
    print("-" * 80)
    stats = comparison_df[['Model', metric]].sort_values(metric, ascending=False)
    display(stats)
    
    # Calculate statistics
    mean = comparison_df[metric].mean()
    std = comparison_df[metric].std()
    print(f"\nMean: {mean:.3f} | Std Dev: {std:.3f}")
    print(f"Range: [{comparison_df[metric].min():.3f}, {comparison_df[metric].max():.3f}]")

## 10. Generate Summary Report

In [ ]:
# Create summary report
report_path = results_dir / 'EVALUATION_REPORT.md'

report = f"""# Multi-Model RAG Evaluation Report

**Date:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Test Dataset:** {len(dataset)} questions
**Models Evaluated:** {len(EMBEDDING_MODELS)}
**Ragas Evaluator:** gpt-4o-mini

## Summary

### Best Overall Model
**{best_overall}** with an average score of **{best_overall_score:.3f}**

### Best by Metric
"""

for metric in metrics:
    best_idx = comparison_df[metric].idxmax()
    best_model = comparison_df.loc[best_idx, 'Model']
    best_score = comparison_df.loc[best_idx, metric]
    report += f"- **{metric}**: {best_model} ({best_score:.3f})\n"

report += f"""

## Detailed Results

| Model | Provider | Dimensions | Context Precision | Context Recall | Faithfulness | Answer Relevancy | Overall Avg |
|-------|----------|------------|-------------------|----------------|--------------|------------------|--------------|
"""

for _, row in comparison_df.iterrows():
    report += f"| {row['Model']} | {row['Provider']} | {row['Dimensions']} | "
    report += f"{row['Context Precision']:.3f} | {row['Context Recall']:.3f} | "
    report += f"{row['Faithfulness']:.3f} | {row['Answer Relevancy']:.3f} | "
    report += f"{row['Overall Average']:.3f} |\n"

report += f"""

## Files Generated

- `comparison_table.csv` - Numeric comparison table
- `comparison_chart.html` - Interactive bar chart
- `radar_chart.html` - Interactive radar chart
- Individual model results in `<model_name>_results.csv`

## Recommendations

Based on the evaluation results:

1. **For Production Use:** Consider {best_overall} for balanced performance
2. **For Cost Optimization:** Review API costs vs performance trade-offs
3. **For Specific Use Cases:** Choose model based on priority metric

## Next Steps

- Analyze per-question performance to identify edge cases
- Consider A/B testing top 2 models in production
- Monitor real-world performance metrics
"""

# Save report
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)

print(f"\n📄 Generated evaluation report: {report_path}")
print("\n" + "="*100)
print(report)
print("="*100)

## Summary

✅ **Evaluation Complete!**

All results have been saved to the results directory. Review:
- Comparison table to see numeric scores
- Interactive charts to visualize differences
- Individual model results for detailed analysis
- Summary report for recommendations